# Find a chemical structure in your documents

Upload PDFs / Word files, give a SMILES, get back **which document, which page,
and where on the page** the structure is drawn.

Pipeline (PatCID, *Nature Communications* 15:6532, 2024):
**DECIMER-Segmentation** -> **MolClassifier** -> **MolGrapher / DECIMER** -> **RDKit matching**

Models and the extraction cache are stored on your Google Drive, so the second
run of this notebook skips the downloads and re-uses everything already processed.

> Runtime -> Change runtime type -> **T4 GPU** makes recognition several times
> faster, but the whole pipeline runs on CPU too.


## 1. Install

In [ ]:
#@title Install dependencies (~5-10 min on a fresh runtime)
%%capture
!pip install -q rdkit pymupdf python-docx opencv-python
!pip install -q decimer-segmentation          # segmentation (PatCID's choice)
!apt-get -qq install -y libreoffice-writer    # .docx -> PDF, keeps page numbers

# Recognition engine. MolGrapher is PatCID's choice and ~2x faster on CPU.
!git clone -q https://github.com/DS4SD/MolGrapher.git /content/MolGrapher
!cd /content/MolGrapher && pip install -q -e ".[cpu]" && bash install_paddleocr.sh

# Classification: Clean / Markush / Trash
!git clone -q https://github.com/DS4SD/MolClassifier.git /content/MolClassifier
!pip install -q pycocotools albumentations imantics more-itertools

# This tool
!git clone -q https://github.com/DS4SD/PatCID.git /content/PatCID

In [ ]:
#@title Register the import paths
import sys

# MolClassifier imports `albumentations_transforms` as a top-level module even
# though it lives inside the package, so its inner directory needs to be on the
# path as well as the repo root.
for path in [
    "/content/PatCID",
    "/content/MolClassifier",
    "/content/MolClassifier/mol_classifier",
]:
    if path not in sys.path:
        sys.path.insert(0, path)

import structure_finder
print("structure_finder", structure_finder.__version__)

## 2. Mount Google Drive and download the models

Everything lands in `MyDrive/structure_finder/`. Run this cell once; on later
sessions it finds the weights already there and returns immediately.

In [ ]:
#@title Download weights to Drive (~2 GB the first time)
from structure_finder import resolve_workspace

workspace = resolve_workspace(use_drive=True)
print("Workspace:", workspace.root)

!python -m structure_finder.setup_models --drive --engine molclassifier --engine decimer-seg --engine molgrapher

## 3. Upload your documents

Accepts `.pdf`, `.docx`, `.doc` and image files. You can also skip this cell and
point step 5 at a Drive folder instead.

In [ ]:
#@title Upload PDFs / Word documents
import os, shutil
from google.colab import files

UPLOAD_DIR = "/content/documents"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(UPLOAD_DIR, name))

print(f"\n{len(uploaded)} document(s) ready in {UPLOAD_DIR}:")
for name in sorted(os.listdir(UPLOAD_DIR)):
    print("  ", name)

## 4. Enter the structure you are looking for

In [ ]:
#@title Query
QUERY_SMILES = "CC(=O)Oc1ccccc1C(=O)O"  #@param {type:"string"}

from rdkit import Chem
from rdkit.Chem import Draw

molecule = Chem.MolFromSmiles(QUERY_SMILES)
assert molecule is not None, "That SMILES could not be parsed - check it."
Chem.RemoveStereochemistry(molecule)
print("Canonical (no stereo):", Chem.MolToSmiles(molecule))
print("InChIKey (no stereo): ", Chem.MolToInchiKey(molecule))
Draw.MolToImage(molecule, size=(350, 350))

## 5. Search

The first pass over a document is the slow one (segment + classify + recognise
every page). The result is cached on Drive by file hash, so searching a
different molecule across the same documents afterwards takes milliseconds.

In [ ]:
#@title Run the search
from structure_finder import find_structure, format_summary, save_all
from pathlib import Path

results = find_structure(
    documents=[UPLOAD_DIR],
    smiles=QUERY_SMILES,
    use_drive=True,
    segmenter="decimer",       # decimer | molclassifier | heuristic
    classifier="molclassifier",# molclassifier | none
    recognizer="molgrapher",   # molgrapher | decimer | ensemble
    match_modes=("exact", "connectivity", "tautomer"),
    dpi=300,
)

print(format_summary(results))

## 6. Look at the matches in context

In [ ]:
#@title Show the annotated pages
from IPython.display import Image, display
from pathlib import Path
from structure_finder import save_all

output_dir = Path("/content/drive/MyDrive/structure_finder/outputs/latest")
written = save_all(results, output_dir, annotate=True)

for page_path in written.get("annotated_pages", []):
    print(page_path)
    display(Image(filename=page_path, width=760))

if not written.get("annotated_pages"):
    print("No image hits to annotate. Check the summary above.")

In [ ]:
#@title Hits as a table
import pandas as pd

hits = pd.DataFrame(results["hits"])
display(hits if len(hits) else "No matches.")

if len(hits):
    hits.to_csv("/content/hits.csv", index=False)
    from google.colab import files
    files.download("/content/hits.csv")

## 7. Search another structure (fast — uses the cache)

In [ ]:
#@title Another query over the same documents
SECOND_QUERY = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"  #@param {type:"string"}

results_2 = find_structure(
    documents=[UPLOAD_DIR],
    smiles=SECOND_QUERY,
    use_drive=True,
    match_modes=("exact", "connectivity", "tautomer"),
)
print(format_summary(results_2))

## Notes on interpreting the result

- **A hit is strong evidence. A miss is weak evidence.** The full PatCID chain
  recovers about 54% of drawn structures exactly on its random benchmark and
  about 41% on the deliberately hard one (older documents, non-US offices).
- If you get no hits and expected some, widen the search:
  `recognizer="ensemble"`, add `"similarity"` to `match_modes`, raise `dpi` to
  400, then inspect `structure_search_extractions.jsonl` — it lists every
  structure the pipeline *did* read, so you can see whether the depiction was
  missed by the segmenter or misread by the recognizer.
- **Markush structures** (generic structures with R groups) cannot equal a
  concrete SMILES. They are counted separately in the summary; search their
  scaffold with a SMARTS query instead:
  `find_structure(..., smiles="smarts:c1ccc2[nH]ccc2c1", match_modes=("substructure",))`
